# Autonomous Driving VLM Representation Dynamics & Language Prior Analysis

**Git-Based 1-Click Execution Runner for Kaggle GPU (Tesla T4 x2 / P100)**

Notebook này tự động clone/pull mã nguồn mới nhất từ GitHub của bạn về Kaggle, nạp checkpoint và chạy phân tích tương phản biểu diễn ($D_l$, VIP, TVI) theo bài báo ICLR 2026.

In [ ]:
# =====================================================================
# 1. CẤU HÌNH GITHUB REPOSITORY CỦA BẠN
# =====================================================================
GITHUB_REPO_URL = "https://github.com/<your-username>/understanding_lp.git"
GITHUB_BRANCH = "main"

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH

# Tự động nạp GitHub Token nếu repo của bạn là Private (tùy chọn)
try:
    from kaggle_secrets import UserSecretsClient
    gh_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if gh_token:
        os.environ["GITHUB_TOKEN"] = gh_token
        print("[Auth] Đã nạp GITHUB_TOKEN từ Kaggle Secrets!")
except Exception:
    pass

print(f"Target Repo: {GITHUB_REPO_URL} (Branch: {GITHUB_BRANCH})")

In [ ]:
%%bash
set -e
cd /kaggle/working

# Xử lý token xác thực nếu repo là Private
REPO_URL="$GITHUB_REPO_URL"
if [ -n "$GITHUB_TOKEN" ] && [[ "$REPO_URL" == https://github.com/* ]]; then
    REPO_URL="https://${GITHUB_TOKEN}@${REPO_URL#https://}"
fi

# Clone hoặc Pull cập nhật mới nhất từ GitHub
if [ ! -d "understanding_lp" ]; then
    echo "[Git] Đang clone repository từ $GITHUB_REPO_URL ($GITHUB_BRANCH)..."
    git clone --branch "$GITHUB_BRANCH" "$REPO_URL" understanding_lp
else
    echo "[Git] Thư mục đã tồn tại. Đang pull cập nhật mới nhất..."
    cd understanding_lp
    git checkout "$GITHUB_BRANCH"
    git pull --ff-only
fi

cd /kaggle/working/understanding_lp
echo "[Dependencies] Cài đặt các thư viện cần thiết..."
pip install -q pyyaml scipy matplotlib transformers tqdm
echo "[Environment] Setup môi trường hoàn tất thành công!"

In [ ]:
# =====================================================================
# 2. KIỂM TRA PHẦN CỨNG GPU & PYTORCH TRÊN KAGGLE
# =====================================================================
import torch, platform

print(f"Python: {platform.python_version()} | PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)} (Số lượng: {torch.cuda.device_count()})")

!nvidia-smi

In [ ]:
# =====================================================================
# 3. CẤU HÌNH & TẢI CHECKPOINT TỪ HUGGING FACE
# =====================================================================
HF_REPO_ID = "huyvanzzz/vsa"        # Repo HF chứa checkpoint của bạn
HF_FILENAME = "finetune_best.pt"    # Tên file checkpoint trên HF (.pt hoặc .pth)

import os
from huggingface_hub import hf_hub_download

# Nạp HF_TOKEN từ Kaggle Secrets nếu repo của bạn là Private
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("[Auth] Đã nạp HF_TOKEN từ Kaggle Secrets!")
except Exception:
    pass

CHECKPOINT_PATH = None
if HF_REPO_ID and HF_FILENAME:
    try:
        print(f"[Hugging Face] Đang kết nối tải checkpoint: {HF_REPO_ID}/{HF_FILENAME}...")
        CHECKPOINT_PATH = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=HF_FILENAME,
            token=os.environ.get("HF_TOKEN")
        )
        print(f"[Hugging Face] Checkpoint sẵn sàng: {CHECKPOINT_PATH}")
    except Exception as exc:
        print(f"[Hugging Face] Chưa tải được checkpoint ({exc}).")
        CHECKPOINT_PATH = None

In [ ]:
# =====================================================================
# 4. CẤU HÌNH & TẢI FEATURE CACHE TỪ HUGGING FACE
# =====================================================================
# Tải các vector đặc trưng của 6 camera từ huyvanzzz/cache (features.float16.memmap)
CACHE_HF_REPO_ID = "huyvanzzz/cache"
CACHE_HF_REPO_TYPE = "dataset"

from huggingface_hub import snapshot_download

CACHE_DIR = None
if CACHE_HF_REPO_ID:
    try:
        print(f"[Hugging Face] Đang tải Feature Cache từ {CACHE_HF_REPO_ID}...")
        CACHE_DIR = snapshot_download(
            repo_id=CACHE_HF_REPO_ID,
            repo_type=CACHE_HF_REPO_TYPE,
            token=os.environ.get("HF_TOKEN")
        )
        print(f"[Hugging Face] Feature Cache sẵn sàng tại: {CACHE_DIR}")
    except Exception as exc:
        print(f"[Hugging Face] Chưa tải được Feature Cache ({exc}).")
        CACHE_DIR = None

In [ ]:
# =====================================================================
# 5. TẢI & CHUẨN BỊ TẬP CÂU HỎI LÁI XE (DRIVELM QA TEST SET)
# =====================================================================
# Tải các câu hỏi và câu trả lời lái xe thực tế từ minhdang0901/drivelm-nuscenes-v1-0
# (Tự động dùng HF_TOKEN để xác thực truy cập Private Dataset)
DATA_FILE = "/kaggle/working/data/test_qa.jsonl"
token_flag = f'--token "{os.environ.get("HF_TOKEN")}"' if os.environ.get("HF_TOKEN") else ""

!python /kaggle/working/understanding_lp/data_preparation/prepare_drivelm_qa.py \
    --repo_id "minhdang0901/drivelm-nuscenes-v1-0" \
    --split "test" \
    --output "$DATA_FILE" \
    {token_flag}

import os
if not os.path.exists(DATA_FILE):
    DATA_FILE = None

In [ ]:
# =====================================================================
# 6. CHẠY PIPELINE PHÂN TÍCH 1-CLICK TRÊN GPU
# =====================================================================
# Mẹo cấu hình:
# - Đặt --max_samples 500: Chạy nhanh ~30-45 giây
# - Xóa --max_samples: Chạy toàn bộ tập dữ liệu test

ckpt_arg = f'--checkpoint "{CHECKPOINT_PATH}"' if CHECKPOINT_PATH else ""
cache_arg = f'--cache_dir "{CACHE_DIR}"' if CACHE_DIR else ""
data_arg = f'--data_file "{DATA_FILE}"' if DATA_FILE else ""

!python /kaggle/working/understanding_lp/scripts/kaggle_run_analysis.py \
    --config /kaggle/working/understanding_lp/configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_k49.yaml \
    {ckpt_arg} \
    {cache_arg} \
    {data_arg} \
    --output_dir /kaggle/working/outputs/ad_analysis \
    --batch_size 16 \
    --max_samples 500 \
    --device auto

In [ ]:
# =====================================================================
# 5. HIỂN THỊ BIỂU ĐỒ KHOA HỌC TRỰC TIẾP TRONG NOTEBOOK
# =====================================================================
from pathlib import Path
from IPython.display import Image, display

output_dir = Path("/kaggle/working/outputs/ad_analysis")
curve_img = output_dir / "divergence_curve.png"
multi_img = output_dir / "multi_panel_analysis.png"

if curve_img.exists():
    print("=" * 65)
    print("BIỂU ĐỒ 1: ĐƯỜNG CONG PHÂN KỲ TẦNG D_l (D_T vs D_VT) & TẦNG VIP")
    print("=" * 65)
    display(Image(filename=str(curve_img), width=850))

if multi_img.exists():
    print("=" * 65)
    print("BIỂU ĐỒ 2: ĐỐI CHIẾU PHÂN KỲ TẦNG & TRỌNG SỐ CHÚ Ý THỊ GIÁC (ATTENTION)")
    print("=" * 65)
    display(Image(filename=str(multi_img), width=1000))

In [ ]:
# =====================================================================
# 6. TỔNG HỢP CHỈ SỐ KHOA HỌC (VIP, TVI GAP, LANGUAGE PRIOR RATIO)
# =====================================================================
import json

metrics_file = output_dir / "divergence_metrics.json"
if metrics_file.exists():
    metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
    print("=" * 65)
    print("BÁO CÁO KẾT QUẢ KHOA HỌC (SCIENTIFIC METRICS):")
    print("=" * 65)
    print(f"Mô hình phân tích:          {metrics.get('model_name')}")
    print(f"Tổng số mẫu đánh giá:      {metrics['total_samples']}")
    print(f"Tỷ lệ Language Prior (D_T): {metrics['language_prior_ratio_pct']:.1f}% ({metrics['dt_count']} mẫu)")
    print(f"Tỷ lệ Visual Driven (D_VT): {100.0 - metrics['language_prior_ratio_pct']:.1f}% ({metrics['dvt_count']} mẫu)")
    print(f"Tầng VIP (Điểm hòa trộn):   Layer {metrics['vip_layer_used']} (Auto-detected: Layer {metrics['vip_layer_auto']})")
    print(f"Điểm TVI (Tổng hòa trộn):   {metrics['tvi']['all_mean']:.4f} ± {metrics['tvi']['all_std']:.4f}")
    print(f"Khoảng cách TVI Gap:        {metrics['tvi']['tvi_gap']:+.4f} (D_VT - D_T)")
    print("=" * 65)

zip_file = output_dir / "analysis_results_bundle.zip"
if zip_file.exists():
    print(f"\n[OK] Đã đóng gói bundle kết quả: {zip_file} ({zip_file.stat().st_size} bytes)")
    print("=> Bạn có thể tải file ZIP này trực tiếp từ bảng 'Data > Output' bên phải màn hình Kaggle!")